# **Autograd Engine**

# Micrograd(Scalar)

In [1]:
import os
dir_path = "custom_autograd"

if os.path.isdir(dir_path):
  print(f"{dir_path} already exists")
else:
  os.makedirs("custom_autograd")

In [2]:
%%writefile custom_autograd/engine.py
import math

_TRACK_GRADIENTS = True

class no_grad:
  def __enter__(self):
    global _TRACK_GRADIENTS
    self.prev = _TRACK_GRADIENTS
    _TRACK_GRADIENTS = False

  def __exit__(self, *args):
    global _TRACK_GRADIENTS
    _TRACK_GRADIENTS = self.prev

class Value:
  def __init__(self, data, _children=(), _op=""):
    self.data = float(data) # stores data

    self.grad = 0.0 # stores gradient

    self._prev = set(_children) # set of parent nodes that created current node

    self._op = _op if _TRACK_GRADIENTS else ""# operation used to create current node

    self._backward = lambda: None # buffer backward function that will be replaced


  @staticmethod
  def _autowrap(other):
    return other if isinstance(other, Value) else Value(other)


  def __add__(self, other):
    # 1. Constanct fallback(Autowrapping)
    # check if other is a Value object
    other = self._autowrap(other)

    # 2. Forward Pass:
    out = Value(self.data+other.data, _children=(self, other), _op="+")

    # 3. Local backward function
    def _backward():
      self.grad += out.grad * 1.0
      other.grad += out.grad * 1.0

    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out


  def __radd__(self, other):
    return self + other


  def __mul__(self, other):
    # 1. Autowrapping
    # check if other is instance of Value class
    other = self._autowrap(other)

    # 2. Forward pass
    out = Value(self.data*other.data, _children=(self, other), _op="*")

    # 3. Backward pass
    def _backward():
      self.grad += out.grad * other.data
      other.grad += out.grad * self.data

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __rmul__(self, other):
    return self * other


  def __pow__(self, other):
    assert isinstance(other, (int, float))

    x = self.data
    y = other
    out = Value(x**y, _children=(self,), _op="**")

    def _backward():
      self.grad += out.grad * (y*x**(y-1))

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __rpow__(self, other):
    return self._autowrap(other)**self


  def __sub__(self, other):
    other = self._autowrap(other)

    out = Value(self.data-other.data, _children=(self, other), _op="-")

    def _backward():
      self.grad += out.grad * 1.0
      other.grad += out.grad * (-1.0)

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __rsub__(self, other):
    return self._autowrap(other) - self


  def __truediv__(self, other):
    other = self._autowrap(other)

    x, y = self.data, other.data

    out = Value(x/y, _children=(self, other), _op="/")

    def _backward():
      self.grad += out.grad * 1/y
      other.grad += out.grad * (x*-1.0*(y**-2))

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __rtruediv__(self, other):
    return self._autowrap(other) / self


  def __neg__(self):
    out = Value(-self.data, _children=(self,), _op="neg")

    def _backward():
      self.grad += out.grad * -1.0

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __rneg__(self):
    return -self


  def backward(self):
    # 1. Custom DFS Graph sort
    topo = []
    visited = set()

    def dfs(node):
      if node not in visited:
        visited.add(node)

        for parent in node._prev:
          dfs(parent)
        topo.append(node)
    dfs(self)

    # 2. Set the initial output gradient to 1.0
    self.grad = 1.0

    # 3. Walk the sorted graph in reverse
    for node in reversed(topo):
      node._backward()


  def tanh(self):
    next_h = math.tanh(self.data)
    out = Value(next_h, _children=(self,), _op="tanh")

    def _backward():
      self.grad += out.grad * (1-next_h*next_h)

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def relu(self):
    x = max(0, self.data)
    out = Value(x, _children=(self,), _op="relu")

    def _backward():
      self.grad += out.grad * (1.0 if x>0 else 0.0)

    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __repr__(self):
    # function for custom representation used for debugging.
    return f"Value(data={self.data}, grad={self.grad})"

Writing custom_autograd/engine.py


In [3]:
%%writefile custom_autograd/nn.py
import random
from custom_autograd.engine import Value

class Neuron:
  def __init__(self, nin):
    # Initializing the weights
    self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]

    # Initializing the bias
    self.bias = Value(random.uniform(-1, 1))

  def __call__(self, x):
    # Multiplying every input by it's corresponding weight
    multiplied = [wi*xi for wi, xi in zip(self.weights, x)]

    total_sum = sum(multiplied, self.bias)

    return total_sum.tanh()

  def parameters(self):
    return self.weights + [self.bias]


class Layer:
  def __init__(self, nin, nout):
    # Create a list of 'nout' distinct neurons
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self, x):
    # Passing input list x to every single neuron and making a list of their outputs
    outputs = [neuron(x) for neuron in self.neurons]

    # single neuron - return salar value object, multi neuron - return output list
    return outputs
  def parameters(self):
    # store weights and bais for each neuron
    params = []
    for neuron in self.neurons:
      params.extend(neuron.parameters())
    return params


class MLP:
  def __init__(self, nin, nouts):
    # 1. Combine input dimensions with output dimensions
    sizes = [nin] + nouts

    # 2. Build the layers sequentially by mapping adjacent sizes
    self.layers = []
    for i in range(len(nouts)):
      self.layers.append(Layer(sizes[i], sizes[i+1]))

  def __call__(self, x):
    # Feeds the input vector sequentially through every single layer
    for layer in self.layers:
      x = layer(x)
    return x[0]

  def parameters(self):
    params = []
    for layer in self.layers:
      params.extend(layer.parameters())
    return params

Writing custom_autograd/nn.py


In [4]:
%%writefile custom_autograd/test.py
# testing the scaler autograd
a = Value(2.0)
b = Value(5.0)

c = a + b
print(f"c={c}")
print(f"parents={c._prev}")

d = a + 2.0
print(f"d={d}")

e = 10.0 + b
print(f"e={e}")

print("Simulating the backward pass")
c.grad = 1.0
c.backward()

print(f"Gradient of a={a.grad}")
print(f"fGradient of b={b.grad}")

Writing custom_autograd/test.py


In [5]:
%%writefile custom_autograd/data.py
import random

# Generating synthetic Binary Classification data and splits the data into training and testing datasets

def generate_data(num_samples: int=100, num_features: int=3, low: float=-2.0, high: float=2.0, random_seed: int=None):
  '''
    generates synthetic binary classification data for the model
    Args:
      num_samples = number of samples(default: 100),
      num_features = number of features(default: 3),
      low = float value representing lower bound for data(default: -2.0)
      high = float value representing higher bound for data(default: 2.0)
      random_seed = integer value to set the random seed(default: None)

    Example:
      generate_data(num_samples=1000, num_features=5, low=-2.0, high=2.0, random_seed=42)
  '''
  if random_seed:
    random.seed(42)

  data = []
  labels = []
  for sample in range(num_samples):
    row = [random.uniform(low, high) for _ in range(num_features)]
    data.append(row)

    if sum(row) > 0.0:
      labels.append(1)
    else:
      labels.append(-1)

  return data, labels

def split_data(data: list=[], labels: list= [], split_value: float=0.8):
  '''
    Splits the data into training and testing datasets and returns them
    Args:
      data = list containing data
      split_value = float value to split data(0.0 to 1.0)

    Example:
      split_data(data=data_list, split_value=0.8)
  '''
  if split_value < 0.0 or split_value > 1.0:
    raise Exception("split_value must be between 0.0 and 1.0")
    return

  n = int(split_value*len(data))
  train_data, train_labels = data[:n], labels[:n]
  test_data, test_labels = data[n:], labels[n:]

  return train_data, train_labels, test_data, test_labels

Writing custom_autograd/data.py


In [6]:
%%writefile custom_autograd/train.py

import argparse
from custom_autograd.engine import no_grad
from custom_autograd.nn import MLP
from custom_autograd.data import generate_data, split_data
from tqdm.auto import tqdm

# Training Custom Binary Classification Model

def main():
  # 1. Parser for command line interface
  parser = argparse.ArgumentParser(description="Train an OO-Autograd Multi-Layer Perceptron on a Hyperplane Classification Task.")

  parser.add_argument("--samples", type=int, default=100, help="Total number of samples in the dataset")
  parser.add_argument("--features", type=int, default=3, help="Total number of features per sample")
  parser.add_argument("--low", type=float, default=-2.0, help="Lower bound for per sample value")
  parser.add_argument("--high", type=float, default=2.0, help="Upper bound for per sample value")
  parser.add_argument("--seed", type=int, default=42, help="Random seed anchor for execution reproducibility")
  parser.add_argument("--split", type=float, default=0.8, help="Train/Test dataset partition split ratio")
  parser.add_argument("--epochs", type=int, default=300, help="Number of optimization training epochs")
  parser.add_argument("--lr", type=float, default=0.01, help="Initial learning rate parameter")
  parser.add_argument("--lr_decay", type=float, default=0.99, help="Learning rate decay parameter")

  args = parser.parse_args()

  NUM_SAMPLES = args.samples
  NUM_FEATURES = args.features
  low, high = args.low, args.high
  RANDOM_SEED = args.seed

  SPLIT_VALUE = args.split

  epochs = args.epochs
  lr = args.lr
  lr_decay = args.lr_decay

  print(f"Initializing (Samples={NUM_SAMPLES}, Epochs={epochs}, Initial learning rate={lr})")

  # 1. Generating the data
  data, labels = generate_data(num_samples=NUM_SAMPLES, num_features=NUM_FEATURES, low=low, high=high, random_seed=RANDOM_SEED)

  # 2. Split data into training and testing datasets
  train_data, train_labels, test_data, test_labels = split_data(data=data, labels=labels, split_value=SPLIT_VALUE)

  # 3. Initialize the model
  model = MLP(NUM_FEATURES, [4, 4, 1])

  for epoch in tqdm(range(epochs)):
    # Forward pass
    y_logits = [model(x) for x in train_data]
    y_pred_labels = [1 if logit.data>=0.0 else -1 for logit in y_logits]

    squared_errors = [(yout-ygt)**2 for ygt, yout in zip(train_labels, y_logits)]

    loss = squared_errors[0]
    for l in squared_errors[1:]:
      loss = loss + l

    # Backward pass
    for p in model.parameters():
      p.grad = 0.0
    loss.backward()

    # Update weights
    for p in model.parameters():
      p.data = p.data + (-lr * p.grad)

    lr = lr * lr_decay

    train_loss_normalized = loss.data / len(train_data)
    total_correct = sum([pred_label==true_label for pred_label, true_label in zip(y_pred_labels, train_labels)])
    accuracy = total_correct * 100 / len(train_labels)

    with no_grad():
      pred_logits = [model(y) for y in test_data]
      pred_labels = [1 if logit.data>=0.0 else -1 for logit in pred_logits]

      squared_pred_errors = [(yout-ygt)**2 for ygt, yout in zip(test_labels, pred_logits)]

      test_loss = squared_pred_errors[0]
      for tl in squared_pred_errors[1:]:
        test_loss = test_loss + tl

      test_loss = test_loss.data / len(test_data)
      test_correct = sum((pred_label==test_label) for pred_label, test_label in zip(pred_labels, test_labels))
      test_accuracy = test_correct * 100/len(test_labels)


    if epoch%10 == 0 or epoch == epochs-1:
      sample_grad = model.parameters()[0].grad
      print(f"Epoch {epoch} | Train loss: {train_loss_normalized:.4f} | Train Accuracy: {accuracy:.1f}% | Sample Grad: {sample_grad:.6f} | Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.1f}")

if __name__ == "__main__":
  main()

Writing custom_autograd/train.py
